In [5]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
import pickle


In [6]:
################ Load model results
# Demand model results
with open('demand_model_results.pkl', 'rb') as f:
     demand_param = pickle.load(f)

# Charging station model results
with open('charging_station_model_results.pkl', 'rb') as f:
     charging_param = pickle.load(f)

# Import data
df = pd.read_csv('demand_counterfactual.csv')
supply = pd.read_csv('supply_counterfactual.csv')

# Add EV_stock from supply to df if columns exist
if 'year' in df.columns and 'province' in df.columns:
    df = df.merge(supply[['year', 'province', 'EV_stock']], on=['year', 'province'], how='left')
else:
    print("Either 'year' or 'province' column is missing in df. Please check your data.")

# Ensure required columns are present in df by merging from supply if missing
required_cols = ['sub_fix', 'sub_ope', 'time_trend', 'num_models_in_market', 'sales_weighted_avg_range']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
	df = df.merge(
		supply[['year', 'province'] + missing_cols],
		on=['year', 'province'],
		how='left'
	)

In [7]:
################ Compute mean utilities δ_jm for nested logit model
def compute_delta_jm(X_jm, demand_par):
    """
    Compute mean utilities δ_jm for nested logit model:
    δ_jm ≡ β_N * log(N_jm) − α * p_jm + x_jm * β_x + ξ_jm
    
    Parameters:
    -----------
    N_jm : array-like
        Number of charging stations (or other network effect variable)
    p_jm : array-like
        Prices for each product-market
    x_jm : 2D array-like
        Matrix of product characteristics (each row is a product-market)
    beta_N : float
        Coefficient on network effect (log stations)
    alpha : float
        Price coefficient
    beta_x : array-like
        Coefficients for product characteristics
    xi_jm : array-like
        Unobserved product-market characteristics
    
    Returns:
    --------
    numpy.ndarray
        Mean utilities δ_jm for each product-market
    """
    x_jm = X_jm[['net_prices', 'range', 'power', 'battery_capacity', 'is_electric', 'log_charging_stock']].values
    beta_x = demand_par.params[['net_prices', 'range', 'power', 'battery_capacity', 'is_electric', 'log_charging_stock_hat']].values
    xi_jm = demand_par.resids

    # Convert inputs to numpy arrays
    x_jm = np.asarray(x_jm)
    beta_x = np.asarray(beta_x)
    
    # Compute utility
    delta_jm = x_jm @ beta_x + X_jm['is_electric'] * X_jm['log_charging_stock'] * demand_par.params['is_electric:log_charging_stock_hat'] + xi_jm
    
    return delta_jm

In [8]:
################ Nested logit shares
def nested_logit_shares(delta_jm, rho, group_membership, market_ids=None):
    """
    Compute nested logit predicted shares for every product in every market,
    using mean utilities of all product-market pairs.

    Parameters
    ----------
    delta_jm : array-like
        Mean utilities for all product-market pairs (length = number of rows in df)
    rho : float
        Nesting parameter (0 ≤ ρ < 1)
    group_membership : array-like
        Nest assignment for each product-market pair (same length as delta_jm)
    market_ids : array-like or None
        Market assignment for each product-market pair (same length as delta_jm).
        If None, treats all rows as one market.

    Returns
    -------
    s_j : np.ndarray
        Share for each product-market pair (same order as input)
    s_0 : np.ndarray
        Outside good share for each market (same order as unique market_ids)
    """
    import numpy as np
    delta_jm = np.array(delta_jm)
    group_membership = np.array(group_membership)
    if market_ids is None:
        market_ids = np.zeros(len(delta_jm), dtype=int)
    else:
        market_ids = np.array(market_ids)

    s_j = np.zeros_like(delta_jm)
    unique_markets = np.unique(market_ids)
    s_0 = np.zeros(len(unique_markets))

    for i, m in enumerate(unique_markets):
        mask_market = (market_ids == m)
        delta_m = delta_jm[mask_market]
        group_m = group_membership[mask_market]
        unique_groups = np.unique(group_m)
        D_g = {}
        for g in unique_groups:
            mask_g = (group_m == g)
            D_g[g] = np.sum(np.exp(delta_m[mask_g] / (1 - rho)))
        log_ratio = np.zeros_like(delta_m)
        for j, (delta, g) in enumerate(zip(delta_m, group_m)):
            log_ratio[j] = delta / (1 - rho) - rho * np.log(D_g[g])
        sum_exp = np.sum(np.exp(log_ratio))
        s_0[i] = 1 / (1 + sum_exp)
        s_j[mask_market] = np.exp(log_ratio) * s_0[i]

    return s_j, s_0

In [9]:
################ Fixed-point iteration for EV-station equilibrium
def fixed_point_iteration(
    design_matrix_init,
    demand_par,
    charging_par,
    rho,
    group_membership,
    max_iter=100,
    tol=1e-6
):
    """
    Fixed-point iteration for EV-station equilibrium at the market level.
    """
    df = design_matrix_init.copy()
    market_ids = df['market_ids'].values
    unique_markets = np.unique(market_ids)

    # Initialize N and Q_ev for each market (take first occurrence in each market)
    N_market = df.groupby('market_ids')['charging_stations_stock'].first().reindex(unique_markets).values
    Q_ev_market = df.groupby('market_ids')['EV_stock'].first().reindex(unique_markets).values

    history = {'N': [], 'Q_ev': [], 'delta': []}

    for iteration in range(max_iter):
        # Assign current market-level N and Q_ev to all rows
        market_idx_map = {m: i for i, m in enumerate(unique_markets)}
        df['N_market'] = [N_market[market_idx_map[m]] for m in market_ids]
        df['Q_ev_market'] = [Q_ev_market[market_idx_map[m]] for m in market_ids]
        df['charging_stations_stock'] = df['N_market']
        df['EV_stock'] = df['Q_ev_market']
        df['log_charging_stock'] = np.log(df['charging_stations_stock'] + 1)

        # Compute mean utilities
        delta = compute_delta_jm(df, demand_par)

        # Compute shares for each product-market
        s_j, _ = nested_logit_shares(delta, rho, group_membership, market_ids=market_ids)

        # Update EV stock for each market by summing EV sales in that market
        ev_mask = df['is_electric'].values == 1
        market_size = df['market_size'].values if 'market_size' in df.columns else np.ones_like(s_j)
        ev_sales = s_j * market_size * ev_mask
        df['ev_sales'] = ev_sales
        Q_ev_market_new = df.groupby('market_ids')['ev_sales'].sum().reindex(unique_markets).values

        # Update station stock for each market using the supply model
        # Use the first row in each market for covariates (since they are market-level)
        supply_covs = df.drop_duplicates('market_ids').set_index('market_ids').reindex(unique_markets)
        log_N = (
            charging_par.params['log(EV_stock)'] * np.log(Q_ev_market_new + 1)
            + charging_par.params['sub_fix'] * supply_covs['sub_fix'].values
            + charging_par.params['sub_ope'] * supply_covs['sub_ope'].values
            + charging_par.params['time_trend'] * supply_covs['time_trend'].values
        )
        for k in charging_par.params.keys():
            if k.startswith('C(province)'):
                province = k.split('[')[-1].strip(']')
                log_N += charging_par.params[k] * (supply_covs['province'] == province).astype(float).values

        N_market_new = np.exp(log_N)

        # Check convergence
        delta_N = np.max(np.abs(N_market_new - N_market))
        delta_Q = np.max(np.abs(Q_ev_market_new - Q_ev_market))
        history['N'].append(N_market_new.copy())
        history['Q_ev'].append(Q_ev_market_new.copy())
        history['delta'].append(max(delta_N, delta_Q))

        if delta_N < tol and delta_Q < tol:
            print(f"Converged after {iteration+1} iterations")
            break

        # Update for next iteration
        N_market = N_market_new
        Q_ev_market = Q_ev_market_new

    else:
        print(f"Warning: Max iterations ({max_iter}) reached")

    return {
        'N': N_market,
        'Q_ev': Q_ev_market,
        'shares': s_j,
        'history': pd.DataFrame(history)
    }

In [10]:
# Compute status quo from fixed-point iteration
result = fixed_point_iteration(
	df,
	demand_param,
	charging_param,
	demand_param.params['log_sj_g'],
	df['nesting_ids'].values
)

Converged after 40 iterations
